# Default `match/` hard-map composition, one pair

This notebook expands the repository's default FAUST matching path one operation at a time for a single non-self pair. It uses the production data loader, `SpectralMapExtractor`, and geodesic implementation, but keeps the intermediate point maps and matrices visible.

The map direction is always **N → M**: every vertex of the second shape (`N`) stores the index of its predicted match in the first shape (`M`). The default method is `spectral-icp` with one iteration. No dataset-wide benchmark is run.

## 1. Setup and production components

In [ ]:
import os

os.environ.setdefault("PYOPENGL_PLATFORM", "egl")

import sys
from pathlib import Path

def find_project_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "match").is_dir():
            return candidate
    local_fallback = Path("/home/knpob/work/code/atm-dev")
    if (local_fallback / "pyproject.toml").is_file():
        return local_fallback
    raise RuntimeError("Could not find the atm-dev repository root.")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pyvista as pv
import torch
import trimesh

from match.data.articulated_shape_corr import ArticulatedShapeCorrDataset
from match.map_extractor.spectral_map_extractor import (
    SpectralMapExtractor,
    _cal_hard_permute,
    _fmap2pointmap,
    _nn_id,
    _spectral_icp_refine,
    _to_device,
)
from match.metric.geodist import GeodistMetric, _geodist_batch

# These values mirror match/config/run.yaml and its default config groups.
DATA_ROOT = Path("/home/knpob/work/data/shape-corr/FAUST_r")
ARTICULATED_SHAPE_DIR = PROJECT_ROOT / "output/faust_r"
PAIR_INDEX = 1  # test Cartesian product: tr_reg_080 -> tr_reg_081
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_EVECS = 200
N_ICP_ITERS = 1
PYVISTA_BACKEND = "static"  # Change to "html" for rotatable scenes.

pv.set_jupyter_backend(PYVISTA_BACKEND)
pv.global_theme.background = "white"
pv.global_theme.font.color = "black"

required_paths = [DATA_ROOT, ARTICULATED_SHAPE_DIR / "mhr"]
missing = [str(path) for path in required_paths if not path.exists()]
assert not missing, f"Missing required assets: {missing}"

print(f"Repository:   {PROJECT_ROOT}")
print(f"FAUST data:   {DATA_ROOT}")
print(f"Articulation: {ARTICULATED_SHAPE_DIR / 'mhr'}")
print(f"Device:       {DEVICE}")

In [ ]:
def to_numpy(value):
    if torch.is_tensor(value):
        return value.detach().cpu().numpy()
    return np.asarray(value)

def as_polydata(vertices, faces=None):
    vertices = to_numpy(vertices)
    if faces is None:
        return pv.PolyData(vertices)
    faces = to_numpy(faces).astype(np.int64, copy=False)
    pv_faces = np.column_stack([np.full(len(faces), 3), faces]).ravel()
    return pv.PolyData(vertices, pv_faces)

def load_mesh(path):
    mesh = trimesh.load(path, process=False)
    if isinstance(mesh, trimesh.Scene):
        mesh = trimesh.util.concatenate(tuple(mesh.geometry.values()))
    return mesh

def spatial_rgb(vertices):
    xyz = to_numpy(vertices).astype(np.float64)
    lo, hi = xyz.min(axis=0), xyz.max(axis=0)
    rgb = (xyz - lo) / np.maximum(hi - lo, 1e-12)
    return np.round(255 * rgb[:, [0, 2, 1]]).astype(np.uint8)

def add_rgb_mesh(plotter, vertices, faces, rgb, title):
    mesh = as_polydata(vertices, faces)
    mesh.point_data["RGB"] = np.asarray(rgb, dtype=np.uint8)
    plotter.add_mesh(mesh, scalars="RGB", rgb=True, smooth_shading=True)
    plotter.add_title(title, font_size=11)
    plotter.camera_position = "xy"

def geodesic_errors(p2p_n2m, shape_m, shape_n):
    return _geodist_batch(
        dist_x=shape_m["dist"][None],
        corr_x=shape_m["corr"][None],
        corr_y=shape_n["corr"][None],
        p2p_y2x=p2p_n2m[None],
    )[0]

def correspondence_errors_on_n(errors, shape_n):
    values = np.full(len(shape_n["verts"]), np.nan, dtype=np.float32)
    values[to_numpy(shape_n["corr"]).astype(np.int64)] = to_numpy(errors)
    return values

## 2. Load exactly one default FAUST pair

The data component below hardcodes the values in `match/config/data/faust_r.yaml`. `PAIR_INDEX=1` selects the first non-self pair from the test Cartesian product; there is no loop over the dataset.

In [ ]:
data_component = ArticulatedShapeCorrDataset(
    dataset_name="faust_r",
    data_root=str(DATA_ROOT),
    articulated_shape_dir=str(ARTICULATED_SHAPE_DIR),
    articulated_subdir="mhr",
    phase="test",
    return_faces=True,
    return_L=True,
    return_mass=True,
    num_evecs=N_EVECS,
    return_evecs=True,
    return_grad=True,
    return_corr=True,
    return_dist=True,
)
pair_dataset = data_component.pipeline()
first_index, second_index = pair_dataset.combinations[PAIR_INDEX]
pair = pair_dataset[PAIR_INDEX]
shape_m = _to_device(pair["first"], DEVICE)
shape_n = _to_device(pair["second"], DEVICE)

articulated_path_m = pair_dataset.dataset.get_articulated_path(first_index)
articulated_path_n = pair_dataset.dataset.get_articulated_path(second_index)
articulated_mesh_m = load_mesh(articulated_path_m)
articulated_mesh_n = load_mesh(articulated_path_n)

assert shape_m["articulated_verts"] is not None
assert shape_n["articulated_verts"] is not None
assert len(articulated_mesh_m.vertices) == len(shape_m["articulated_verts"])
assert len(articulated_mesh_n.vertices) == len(shape_n["articulated_verts"])

print(f"Pair: {shape_m['name']} (M) <- {shape_n['name']} (N)")
print(f"Original vertices:    M={len(shape_m['verts']):,}, N={len(shape_n['verts']):,}")
print(f"Articulated vertices: M={len(shape_m['articulated_verts']):,}, N={len(shape_n['articulated_verts']):,}")
print(f"Eigenbasis:           M={tuple(shape_m['evecs'].shape)}, N={tuple(shape_n['evecs'].shape)}")

In [ ]:
shared_articulated_topology = (
    len(articulated_mesh_m.vertices) == len(articulated_mesh_n.vertices)
    and np.array_equal(articulated_mesh_m.faces, articulated_mesh_n.faces)
)
assert shared_articulated_topology, (
    "The composition assumes that the two articulated meshes share vertex IDs and faces."
)

plotter = pv.Plotter(shape=(1, 2), window_size=(1400, 650))
for column, (shape, articulated_mesh, label) in enumerate([
    (shape_m, articulated_mesh_m, "M / first shape"),
    (shape_n, articulated_mesh_n, "N / second shape"),
]):
    plotter.subplot(0, column)
    plotter.add_mesh(
        as_polydata(shape["verts"], shape["faces"]),
        color="white", opacity=0.45, label="original", smooth_shading=True,
    )
    plotter.add_mesh(
        pv.wrap(articulated_mesh),
        color="salmon", opacity=0.55, label="articulated", smooth_shading=True,
    )
    plotter.add_title(f"{label}: original/articulated overlay", font_size=11)
    plotter.camera_position = "xy"
plotter.link_views()
plotter.add_legend()
plotter.show()

## 3. Spatial hard-map composition

The production `coarse_map()` performs two Euclidean nearest-neighbor queries and composes their indices:

`N original → N articulated == M articulated → M original`

The middle equality is an implicit identity map on the shared articulated-template vertex IDs. This is the key assumption to inspect visually.

In [ ]:
map_extractor = SpectralMapExtractor(
    device=DEVICE,
    method="spectral-icp",
    n_iters=N_ICP_ITERS,
)

# N original -> N articulated template.
dist_n2a, p2p_n2a_np = _nn_id(shape_n["verts"], shape_n["articulated_verts"])
# M articulated template -> M original.
dist_a2m, p2p_a2m_np = _nn_id(shape_m["articulated_verts"], shape_m["verts"])

p2p_n2a = torch.as_tensor(p2p_n2a_np, device=DEVICE, dtype=torch.long)
p2p_a2m = torch.as_tensor(p2p_a2m_np, device=DEVICE, dtype=torch.long)
p2p_coarse = p2p_a2m[p2p_n2a]
p2p_coarse_production = map_extractor.coarse_map(shape_m, shape_n)
assert torch.equal(p2p_coarse, p2p_coarse_production)

print("Manual composition exactly matches SpectralMapExtractor.coarse_map().")
print(f"N -> articulated NN: mean={dist_n2a.mean():.6f}, p95={np.quantile(dist_n2a, 0.95):.6f}, max={dist_n2a.max():.6f}")
print(f"articulated -> M NN: mean={dist_a2m.mean():.6f}, p95={np.quantile(dist_a2m, 0.95):.6f}, max={dist_a2m.max():.6f}")

In [ ]:
# Pull M's spatial RGB field backward through each lookup in the N -> M map.
rgb_m = spatial_rgb(shape_m["verts"])
rgb_articulated = rgb_m[p2p_a2m_np]
rgb_n = rgb_articulated[p2p_n2a_np]

plotter = pv.Plotter(shape=(1, 4), window_size=(2000, 520))
panels = [
    (shape_n["verts"], shape_n["faces"], rgb_n, "N original (query)"),
    (articulated_mesh_n.vertices, articulated_mesh_n.faces, rgb_articulated, "N articulated"),
    (articulated_mesh_m.vertices, articulated_mesh_m.faces, rgb_articulated, "M articulated"),
    (shape_m["verts"], shape_m["faces"], rgb_m, "M original (lookup)"),
]
for column, panel in enumerate(panels):
    plotter.subplot(0, column)
    add_rgb_mesh(plotter, *panel)
plotter.link_views()
plotter.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
axes[0].hist(dist_n2a, bins=80, alpha=0.8, label="N original → articulated")
axes[0].hist(dist_a2m, bins=80, alpha=0.6, label="articulated → M original")
axes[0].set_xlabel("Euclidean nearest-neighbor distance")
axes[0].set_ylabel("vertex count")
axes[0].set_title("Distances used by the coarse map")
axes[0].legend()

axes[1].scatter(
    np.arange(len(p2p_coarse)), to_numpy(p2p_coarse), s=0.35, alpha=0.7, rasterized=True
)
axes[1].set_xlabel("vertex ID in N")
axes[1].set_ylabel("matched vertex ID in M")
axes[1].set_title("Composed hard point map N → M")
plt.show()

## 4. Hard permutation and functional-map lift

`Pmn[m, n] = 1` exactly when N vertex `n` maps to M vertex `m`. Every column must contain one entry; rows may contain zero or several because nearest-neighbor maps need not be bijective. The default extractor then computes

`Cmn = Φnᵀ A_n Pmnᵀ Φm`

using the mass-weighted transposed eigenvectors stored as `evecs_trans`.

In [ ]:
pmn = _cal_hard_permute(shape_m, shape_n, p2p_coarse)
column_sums = pmn.sum(dim=0)
row_sums = pmn.sum(dim=1)
assert torch.all(column_sums == 1), "Each N vertex must map to exactly one M vertex."

cmn = shape_n["evecs_trans"] @ pmn.T @ shape_m["evecs"]
p2p_spectral_nn = _fmap2pointmap(cmn, shape_m["evecs"], shape_n["evecs"])

# A compact occupancy image of Pmn; no dense matrix is copied back for plotting.
preview_size = 512
preview = np.zeros((preview_size, preview_size), dtype=np.uint8)
preview_rows = np.minimum(
    preview_size - 1, to_numpy(p2p_coarse) * preview_size // len(shape_m["verts"])
).astype(np.int64)
preview_cols = np.arange(len(p2p_coarse)) * preview_size // len(p2p_coarse)
preview[preview_rows, preview_cols] = 1

fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
axes[0].imshow(preview, origin="lower", cmap="gray_r", interpolation="nearest")
axes[0].set_xlabel("N vertex bins")
axes[0].set_ylabel("M vertex bins")
axes[0].set_title("Pmn occupancy (512 × 512 binned)")
image = axes[1].imshow(to_numpy(cmn).T, cmap="coolwarm", aspect="equal")
axes[1].set_xlabel("N basis index")
axes[1].set_ylabel("M basis index")
axes[1].set_title("Initial functional map Cmnᵀ")
fig.colorbar(image, ax=axes[1], shrink=0.8)
plt.show()

print(f"Pmn shape: {tuple(pmn.shape)}")
print(f"Occupied M rows: {int((row_sums > 0).sum()):,} / {len(row_sums):,}")
print(f"Maximum collisions in one M row: {int(row_sums.max())}")
print(f"Cmn shape: {tuple(cmn.shape)}")

## 5. The default one-step spectral ICP refinement

This cell spells out the body of `_spectral_icp_refine()` for the configured single iteration: project M's spectral embedding into N's basis, find nearest N embeddings, solve an orthogonal Procrustes problem, then discretize the refined functional map back to N → M. It checks the result against the production function and `refine_map()`.

In [ ]:
phi_m = shape_m["evecs"].T
phi_n = shape_n["evecs"].T
c_icp = cmn.clone()

for _ in range(N_ICP_ITERS):
    projected_m = c_icp @ phi_m
    spectral_distances = torch.cdist(projected_m.T, phi_n.T)
    matches_m2n = spectral_distances.argmin(dim=1)
    x_tilde = phi_n[:, matches_m2n]
    w = x_tilde @ phi_m.T
    u, _, vt = torch.linalg.svd(w)
    c_icp = u @ vt
    del projected_m, spectral_distances

p2p_icp = _fmap2pointmap(c_icp, shape_m["evecs"], shape_n["evecs"])
p2p_icp_helper = _spectral_icp_refine(
    cmn, shape_m["evecs"], shape_n["evecs"], n_iters=N_ICP_ITERS
)
p2p_default = map_extractor.refine_map(shape_m, shape_n, p2p_coarse)
assert torch.equal(p2p_icp, p2p_icp_helper)
assert torch.equal(p2p_icp, p2p_default)

identity = torch.eye(c_icp.shape[0], device=c_icp.device, dtype=c_icp.dtype)
orthogonality_before = torch.linalg.norm(cmn @ cmn.T - identity) / np.sqrt(len(cmn))
orthogonality_after = torch.linalg.norm(c_icp @ c_icp.T - identity) / np.sqrt(len(c_icp))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)
limit = max(float(cmn.abs().max()), float(c_icp.abs().max()))
for ax, matrix, title in [
    (axes[0], cmn, "Before ICP: Cmn"),
    (axes[1], c_icp, "After one ICP step"),
]:
    image = ax.imshow(to_numpy(matrix), cmap="coolwarm", vmin=-limit, vmax=limit)
    ax.set_xlabel("M basis index")
    ax.set_ylabel("N basis index")
    ax.set_title(title)
fig.colorbar(image, ax=axes, shrink=0.8)
plt.show()

print("Manual ICP result exactly matches both production code paths.")
print(f"Orthogonality residual: {orthogonality_before.item():.6f} -> {orthogonality_after.item():.6f}")

## 6. Compare the three visible maps

The default output is the last map. The intermediate functional-map nearest-neighbor result is included only to reveal what the ICP step changes. Colors are generated on M and pulled onto N by each N → M point map.

In [ ]:
stage_maps = {
    "spatial composition": p2p_coarse,
    "functional-map NN": p2p_spectral_nn,
    "spectral ICP (default)": p2p_icp,
}
stage_errors = {name: geodesic_errors(p2p, shape_m, shape_n) for name, p2p in stage_maps.items()}
mean_errors = {name: float(errors.mean()) for name, errors in stage_errors.items()}

# Exercise the production metric class on the same final map. It does not export without export_path.
metric = GeodistMetric(out_dir=str(ARTICULATED_SHAPE_DIR), export_path=None)
metric_output = metric.pipeline({
    "shape_m": shape_m,
    "shape_n": shape_n,
    "p2p": p2p_icp,
    "method": map_extractor.method,
    "dataset_name": data_component.dataset_name,
})
assert np.isclose(float(metric_output), mean_errors["spectral ICP (default)"], atol=5e-7)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(mean_errors.keys(), mean_errors.values(), color=["#4c78a8", "#f2cf5b", "#e45756"])
ax.bar_label(bars, fmt="%.6f", padding=3)
ax.set_ylabel("mean geodesic error")
ax.set_title(f"One-pair error: {shape_n['name']} → {shape_m['name']}")
ax.tick_params(axis="x", rotation=12)
plt.show()

mean_errors

In [ ]:
plotter = pv.Plotter(shape=(1, 4), window_size=(2000, 520))
plotter.subplot(0, 0)
add_rgb_mesh(plotter, shape_m["verts"], shape_m["faces"], rgb_m, "M source colors")
for column, (name, p2p) in enumerate(stage_maps.items(), start=1):
    plotter.subplot(0, column)
    rgb_transfer = rgb_m[to_numpy(p2p).astype(np.int64)]
    add_rgb_mesh(plotter, shape_n["verts"], shape_n["faces"], rgb_transfer, name)
plotter.link_views()
plotter.show()

In [ ]:
error_fields = {
    name: correspondence_errors_on_n(errors, shape_n)
    for name, errors in stage_errors.items()
}
finite_errors = np.concatenate([values[np.isfinite(values)] for values in error_fields.values()])
shared_clim = [0.0, float(np.quantile(finite_errors, 0.98))]

plotter = pv.Plotter(shape=(1, 3), window_size=(1650, 560))
for column, (name, values) in enumerate(error_fields.items()):
    plotter.subplot(0, column)
    mesh = as_polydata(shape_n["verts"], shape_n["faces"])
    mesh.point_data["geodesic error"] = values
    plotter.add_mesh(
        mesh, scalars="geodesic error", cmap="magma", clim=shared_clim,
        smooth_shading=True, show_scalar_bar=(column == 2),
    )
    plotter.add_title(f"{name}\nmean={mean_errors[name]:.6f}", font_size=10)
    plotter.camera_position = "xy"
plotter.link_views()
plotter.show()

In [ ]:
diagnostics = {
    "pair_n_to_m": f"{shape_n['name']} -> {shape_m['name']}",
    "device": DEVICE,
    "method": map_extractor.method,
    "icp_iterations": map_extractor.n_iters,
    "shared_articulated_topology": shared_articulated_topology,
    "coarse_matches_production": bool(torch.equal(p2p_coarse, p2p_coarse_production)),
    "icp_matches_production": bool(torch.equal(p2p_icp, p2p_default)),
    "hard_map_columns_with_one_entry": int((column_sums == 1).sum()),
    "hard_map_total_columns": len(column_sums),
    "occupied_m_vertices": int((row_sums > 0).sum()),
    "max_collision_count": int(row_sums.max()),
    "geodesic_errors": mean_errors,
    "production_metric_output": metric_output,
}
diagnostics

## Visual correctness checklist

- In each overlay, the salmon articulated reconstruction should occupy the same pose, scale, orientation, and location as its translucent original.
- In the four-stage composition, colors should remain on the same anatomy across `N original → N articulated == M articulated → M original`; left/right swaps or abrupt color seams indicate a transform or shared-indexing problem.
- Both nearest-neighbor histograms should be concentrated near zero without a large secondary mode.
- Every hard-permutation column must have exactly one entry. Empty M rows and modest row collisions are expected because the map is not forced to be bijective.
- The functional map should have coherent low-frequency structure; the ICP result should be orthogonal up to numerical precision.
- Transferred colors on the final N mesh should preserve body regions without mirrored limbs or isolated speckle.
- Geodesic-error heatmaps should not contain large anatomy-scale hot regions. ICP is not guaranteed to improve every individual pair, so correctness here means matching the production computation and preserving a plausible correspondence—not monotonically reducing this one-pair score.